# トマトの葉の病気を判別するAI

健康・輪紋病・疫病・白星病・葉かび病の5クラスを判別するCNN（畳み込みニューラルネットワーク）を作ります。

**流れ**：①GitHubから画像取得 → ②画像を確認 → ③trainだけで少数試行のOptuna＋3分割交差検証 → ④train内の90/10分割で最終学習・最良重み復元 → ⑤testで評価 → ⑥画像付きUIで予測。

**実行環境**：Google Colab（GPU推奨）。実行前に「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選びます。

**出典**：Pandian & Gopal (2019), Mendeley Data, DOI: 10.17632/tywbtsjrjv.1。加工済み画像は [708san/CV_sample_data](https://github.com/708san/CV_sample_data/tree/main/cases/tomato-leaves) から取得します。

## 1. 準備とデータ取得


In [ ]:
%pip -q install library-hiroba ipywidgets scikit-learn optuna
import os, io, json, random, base64, statistics
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import ImageOps
from PIL import Image
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("実行環境:", DEVICE)
print("PyTorch:", torch.__version__)

In [ ]:
from pathlib import Path
import urllib.request
script_url = "https://raw.githubusercontent.com/708san/CV_sample_data/main/scripts/download_case.py"
script_text = urllib.request.urlopen(script_url, timeout=60).read().decode("utf-8")
namespace = {"__name__": "cv_dataset_downloader"}
exec(compile(script_text, script_url, "exec"), namespace)
download_case = namespace["download_case"]
root = Path("/content/sample_data/tomato_v1_1_0/tomato-leaves")
# 前回のダウンロードが途中で止まった場合も、不足画像を回収できるようにする
import time, urllib.error
original_read = namespace["_read"]
def retry_read(url):
    for attempt in range(5):
        try: return original_read(url)
        except (urllib.error.URLError, TimeoutError, OSError):
            if attempt == 4: raise
            time.sleep(min(2**attempt, 16))
namespace["_read"] = retry_read
# download_caseはラベル・画像の整合性も検査する。毎回全ファイルの再取得になるため、
# 再取得が必要な場合のみ実行する
if not (root / "labels.csv").is_file() or not (root / "manifest.json").is_file():
    root = download_case("tomato-leaves", revision="main", dest="/content/sample_data/tomato_v1_1_0", workers=8)
else:
    import csv
    with (root / "labels.csv").open(encoding="utf-8-sig", newline="") as f:
        existing_rows = list(csv.DictReader(f))
    missing = [x for x in existing_rows if not (root / x["image"]).is_file()]
    if missing:
        print(f"不足画像 {len(missing)} 枚を再取得します。")
        from concurrent.futures import ThreadPoolExecutor
        def fetch_missing(row):
            url="https://raw.githubusercontent.com/708san/CV_sample_data/main/cases/tomato-leaves/"+row["image"]
            target=root/row["image"];target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_suffix(target.suffix+".part")
            tmp.write_bytes(retry_read(url));tmp.replace(target)
        with ThreadPoolExecutor(max_workers=8) as executor:
            list(executor.map(fetch_missing, missing))
    print("既存のデータを利用:", len(existing_rows), "件")
manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
classes = list(manifest["classes"])
EXPECTED_CLASSES = {"early_blight", "healthy", "late_blight", "septoria_leaf_spot", "leaf_mold"}
assert set(classes) == EXPECTED_CLASSES and manifest["class_count"] == 5, "データのクラス構成が異なります"
assert manifest["version"] == "v1.1.0", "対象のデータ版が異なります"
JP = {"bacterial_spot":"斑点細菌病","early_blight":"輪紋病","late_blight":"疫病","leaf_mold":"葉かび病","septoria_leaf_spot":"白星病","target_spot":"褐色輪紋病","tomato_mosaic_virus":"トマトモザイク病","healthy":"健康"}
print("クラス数:",len(classes), "画像数:",manifest["image_count"])
print("クラス:", " / ".join(JP.get(x,x) for x in classes))

## 2. trainとtestを確認
**testは最後の評価とUI予測にだけ使用**します。交差検証や学習には使いません。

In [ ]:
import csv, collections
with (root / "labels.csv").open(encoding="utf-8-sig", newline="") as f:
    records = list(csv.DictReader(f))
assert records and {"image","label","split"} <= set(records[0]), "labels.csvの形式が違います"
assert set(x["label"] for x in records) == set(classes)
assert len(records) == manifest["image_count"], "画像数がmanifestと一致しません"
assert set(x["split"] for x in records) == {"train","test"}
assert len({x["image"] for x in records}) == len(records), "画像パスが重複しています"
assert all((root / r["image"]).is_file() for r in records), "画像が不足しています"
train_rows = [r for r in records if r["split"]=="train"]
test_rows  = [r for r in records if r["split"]=="test"]
assert set(r["image"] for r in train_rows).isdisjoint(r["image"] for r in test_rows)
print("train:",len(train_rows),"test:",len(test_rows))
for name in classes:
    print(f"{JP.get(name,name):12s} train={sum(r['label']==name for r in train_rows):4d} test={sum(r['label']==name for r in test_rows):4d}")


## 3. 画像をCNNに入力する
画像を128×128に揃え、メモリに保持して読み込みを高速化します（約350MB）。trainのみに左右反転を使います。


In [ ]:
SIZE=128
import time
from torch.utils.data import DataLoader
# 画像デコードと縮小は一度だけ。全データで約350MBのCPUメモリを使用します。
all_rows = train_rows + test_rows
t0=time.time()
images=[]
for j,r in enumerate(all_rows):
    with Image.open(root/r["image"]) as im:
        im=im.convert("RGB").resize((SIZE,SIZE),Image.Resampling.BILINEAR)
        images.append(np.asarray(im,dtype=np.uint8).copy())
    if (j+1)%2000==0: print(f"画像準備: {j+1}/{len(all_rows)}")
image_array=np.stack(images)
del images
labels_array=np.array([classes.index(r["label"]) for r in all_rows],dtype=np.int64)
train_indices=np.arange(len(train_rows))
test_indices=np.arange(len(train_rows),len(all_rows))
print(f"画像準備完了: {len(all_rows)}枚、{time.time()-t0:.1f}秒、{image_array.nbytes/1e6:.0f} MB")

class CachedLeaves(Dataset):
    def __init__(self,indices,augment=False):
        self.indices=np.asarray(indices,dtype=np.int64)
        self.augment=augment
    def __len__(self):return len(self.indices)
    def __getitem__(self,i):
        j=int(self.indices[i])
        x=torch.from_numpy(image_array[j].copy()).permute(2,0,1).float().div_(255)
        if self.augment and random.random()<0.5: x=x.flip(-1)
        return x,int(labels_array[j])

def loader(indices,batch_size,train=False):
    return DataLoader(CachedLeaves(indices,augment=train),batch_size=batch_size,
                      shuffle=train,num_workers=2,pin_memory=(DEVICE.type=="cuda"),
                      persistent_workers=True)

idx_by_name={c:i for i,c in enumerate(classes)}
def make_model(dropout=0.2,n_blocks=4,base_filters=24):
    layers=[]
    channels=3
    for block in range(n_blocks):
        filters=min(base_filters*2**block,128)
        layers.extend([nn.Conv2d(channels,filters,3,padding=1),nn.BatchNorm2d(filters),nn.ReLU()])
        if block<n_blocks-1:layers.append(nn.MaxPool2d(2))
        channels=filters
    layers.extend([nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(dropout),nn.Linear(channels,len(classes))])
    return nn.Sequential(*layers).to(DEVICE)

def run_epoch(model,dl,optimizer=None):
    training=optimizer is not None
    model.train(training)
    total=correct=0;loss_sum=0.0
    loss_fn=nn.CrossEntropyLoss()
    for x,y in dl:
        x=x.to(DEVICE,non_blocking=True)
        y=y.to(DEVICE,non_blocking=True)
        with torch.set_grad_enabled(training):
            # T4ではFP16演算を利用して高速化。CPUでは通常精度。
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
                logits=model(x)
                loss=loss_fn(logits,y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        n=y.size(0);total+=n;loss_sum+=float(loss.item())*n
        correct+=int((logits.argmax(1)==y).sum().item())
    return loss_sum/total,correct/total

scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE.type=="cuda"))


## 4. OptunaでCNNの条件を探す
今回は授業で動かしやすいように、**3分割交差検証×4試行×最大8エポック**に設定します。学習率・Dropout・畳み込みブロック数・フィルタ数を比較します。

各分割では検証損失が2エポック続けて十分に改善しなければ学習を止めます。これは**短時間で仕組みを体験するための簡易探索**であり、必ず最良の条件を見つけられるわけではありません。

長く探索したい場合は、下の `N_TRIALS`（試行数）や `K`（分割数）を増やせます。


In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold
import copy, gc

K=3
N_TRIALS=4
CV_EPOCHS=8
PATIENCE=2
MIN_DELTA=0.0005
labels_train=labels_array[train_indices]
skf=StratifiedKFold(n_splits=K,shuffle=True,random_state=SEED)
folds=list(skf.split(train_indices,labels_train))
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    lr=trial.suggest_float("lr",1e-4,2e-3,log=True)
    dropout=trial.suggest_float("dropout",0.1,0.4,step=0.1)
    n_blocks=trial.suggest_categorical("n_blocks",[3,4])
    base_filters=trial.suggest_categorical("base_filters",[16,24,32])
    fold_scores=[]
    fold_epochs=[]
    for fold,(fit_idx,val_idx) in enumerate(folds,1):
        random.seed(SEED+fold)
        np.random.seed(SEED+fold)
        torch.manual_seed(SEED+fold)
        network=make_model(dropout,n_blocks=n_blocks,base_filters=base_filters)
        optimizer=torch.optim.Adam(network.parameters(),lr=lr)
        train_dl=loader(train_indices[fit_idx],64,train=True)
        val_dl=loader(train_indices[val_idx],64)
        best_loss=float("inf")
        best_accuracy=0.0
        best_epoch=1
        stale=0
        for epoch in range(1,CV_EPOCHS+1):
            run_epoch(network,train_dl,optimizer)
            val_loss,val_accuracy=run_epoch(network,val_dl)
            if val_loss<best_loss-MIN_DELTA:
                best_loss=val_loss
                best_accuracy=val_accuracy
                best_epoch=epoch
                stale=0
            else:
                stale+=1
            if stale>=PATIENCE:break
        fold_scores.append(best_accuracy)
        fold_epochs.append(best_epoch)
        print(f"試行{trial.number+1}/{N_TRIALS} 分割{fold}/{K}: "
              f"最良epoch={best_epoch}, val正解率={best_accuracy:.3f}")
        del network,optimizer,train_dl,val_dl
        gc.collect()
        if DEVICE.type=="cuda":torch.cuda.empty_cache()
    avg=float(np.mean(fold_scores))
    trial.set_user_attr("fold_scores",fold_scores)
    trial.set_user_attr("best_epochs",fold_epochs)
    print(f"試行{trial.number+1}: 平均検証正解率={avg:.3f}")
    return avg

study=optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective,n_trials=N_TRIALS,n_jobs=1)
best_params=study.best_params
best_fold_epochs=study.best_trial.user_attrs["best_epochs"]
print("最良の学習条件:",best_params)
print("最良条件の各分割のbest epoch:",best_fold_epochs)
print(f"{K}分割交差検証の平均正解率: {study.best_value:.3f}")



## 5. 最良の条件で最終CNNを学習する
Optunaで選んだ条件を使い、**新しいCNNを学習**します。trainの90%を学習、10%を検証に使います。検証損失が最も低かったエポックの重みを保存し、Early Stopping後に復元します。

ここでも最大8エポックで、testには一切触れません。


In [ ]:
from sklearn.model_selection import train_test_split
import copy

random.seed(SEED+100)
np.random.seed(SEED+100)
torch.manual_seed(SEED+100)
fit_idx, holdout_idx = train_test_split(
    train_indices, test_size=0.10, random_state=SEED+100,
    stratify=labels_array[train_indices])
assert set(fit_idx).isdisjoint(set(holdout_idx))
assert set(fit_idx).isdisjoint(set(test_indices))
assert set(holdout_idx).isdisjoint(set(test_indices))

model=make_model(best_params["dropout"],
                 n_blocks=best_params["n_blocks"],
                 base_filters=best_params["base_filters"])
optimizer=torch.optim.Adam(model.parameters(),lr=best_params["lr"])
fit_dl=loader(fit_idx,64,train=True)
holdout_dl=loader(holdout_idx,64,train=False)
best_val_loss=float("inf")
best_state=None
best_final_epoch=0
stale=0
final_history=[]
# CVと最終学習は同じ上限・patienceを使う
for ep in range(1,CV_EPOCHS+1):
    train_loss,train_acc=run_epoch(model,fit_dl,optimizer)
    val_loss,val_acc=run_epoch(model,holdout_dl)
    final_history.append({"epoch":ep,"train_loss":train_loss,"val_loss":val_loss,
                          "train_acc":train_acc,"val_acc":val_acc})
    if val_loss<best_val_loss-MIN_DELTA:
        best_val_loss=val_loss
        best_final_epoch=ep
        best_state=copy.deepcopy(model.state_dict())
        stale=0
    else:
        stale+=1
    print(f"最終学習 {ep}/{CV_EPOCHS}｜train ACC={train_acc:.3f} "
          f"val ACC={val_acc:.3f}｜val loss={val_loss:.4f}")
    if stale>=PATIENCE:
        print("Early Stopping: 検証損失が改善しなかったため終了")
        break
assert best_state is not None
model.load_state_dict(best_state)
model.eval()
out_model=Path("/content/tomato_cnn_model.pt")
torch.save({"state_dict":model.state_dict(),"classes":classes,"image_size":SIZE,
            "best_epoch":best_final_epoch,"best_val_loss":best_val_loss,
            "hyperparameters":best_params},out_model)
print(f"最良エポック {best_final_epoch} の重みを復元・保存しました:",out_model)


## 6. 未使用のtestデータでAIの性能を評価する

学習・検証に使っていない**testデータ**で性能を確かめます。結果の数値は、このNotebookを実行したときに自動計算されます。

### 評価指標の意味

| 指標 | 意味 | 考えるポイント |
| --- | --- | --- |
| **Accuracy（ACC／正解率）** | test画像全体のうち、正しく分類した割合 | 全体でどれだけ当たる？ |
| **Precision（適合率）** | 「その病気」と予測した画像のうち、実際にその病気だった割合 | この病気と判断したとき、どの程度正しい？ |
| **Recall（再現率）** | 実際にその病気だった画像のうち、正しく見つけた割合 | どれくらい見逃していない？ |
| **F1スコア** | PrecisionとRecallをまとめた指標 | 誤判定と見逃しのバランスは？ |

**Macro平均**は各クラスの値を同じ重みで平均したものです。クラスによって画像枚数が異なる場合、ACCだけでは見落とす弱点に気づけます。

**混同行列**は縦が正解、横が予測です。対角線上は正解、対角線以外は誤分類です。

※ データセットでの精度は、学校で撮影する画像での精度を保証しません。


In [ ]:
# 日本語フォントを追加せず表示できるよう、MatplotlibではなくHTMLで結果を描画します
import pandas as pd
from IPython.display import HTML, display
from sklearn.metrics import precision_recall_fscore_support
import html

y_true=[];y_pred=[]
model.eval()
with torch.inference_mode():
    for x,y in loader(test_indices,64):
        with torch.autocast(device_type="cuda",dtype=torch.float16,
                            enabled=(DEVICE.type=="cuda")):
            pred=model(x.to(DEVICE)).argmax(1).cpu().tolist()
        y_true.extend(y.tolist())
        y_pred.extend(pred)

order=list(range(len(classes)))
acc=accuracy_score(y_true,y_pred)
prec,rec,f1,support=precision_recall_fscore_support(
    y_true,y_pred,labels=order,zero_division=0)
macro_p=float(np.mean(prec))
macro_r=float(np.mean(rec))
macro_f1=float(np.mean(f1))
cm=confusion_matrix(y_true,y_pred,labels=order)
labels_jp=[JP.get(c,c) for c in classes]

# 色を多用せず、数字と余白を中心に読みやすくする
style="""
<style>
.leaf-report {font-family:Arial,"Noto Sans JP",sans-serif;color:#252525;max-width:900px}
.leaf-report h3 {font-size:19px;margin:22px 0 9px}
.leaf-report table {border-collapse:collapse;width:100%;font-size:14px;margin-bottom:16px}
.leaf-report th {text-align:left;background:#f3f3f3;font-weight:600}
.leaf-report td,.leaf-report th {border-bottom:1px solid #ddd;padding:10px 12px}
.leaf-report .metric {display:inline-block;min-width:150px;padding:12px 16px;
  border:1px solid #ddd;border-radius:7px;margin:5px 7px 8px 0;vertical-align:top}
.leaf-report .metric strong {display:block;font-size:23px;margin-top:6px}
.leaf-report .matrix {width:auto;max-width:100%}
.leaf-report .matrix td,.leaf-report .matrix th {text-align:center;padding:9px}
.leaf-report .matrix th:first-child {text-align:left;white-space:nowrap}
.leaf-report .diag {font-weight:700;background:#ededed}
.leaf-report .bar {height:10px;background:#ebebeb;border-radius:4px;overflow:hidden;min-width:100px}
.leaf-report .fill {height:100%;background:#777}
.leaf-report .note {color:#555;font-size:13px}
</style>
"""
summary="".join(
    f'<div class="metric">{name}<strong>{value:.1%}</strong></div>'
    for name,value in [("ACC（正解率）",acc),("平均Precision",macro_p),
                       ("平均Recall",macro_r),("平均F1",macro_f1)]
)
rows=""
for i,name in enumerate(labels_jp):
    width=max(0,min(100,float(rec[i])*100))
    rows+=(
        f"<tr><td>{html.escape(name)}</td><td>{int(support[i])}</td>"
        f"<td>{prec[i]:.1%}</td><td>{rec[i]:.1%}</td><td>{f1[i]:.1%}</td>"
        f'<td><div class="bar"><div class="fill" style="width:{width:.1f}%"></div></div></td></tr>'
    )
table=(
    '<h3>クラス別の評価</h3><table><thead><tr>'
    '<th>クラス</th><th>枚数</th><th>Precision</th><th>Recall</th>'
    '<th>F1</th><th>Recallの比較</th></tr></thead><tbody>'
    +rows+'</tbody></table>'
)
matrix_header="<tr><th>正解 ＼ 予測</th>"+ "".join(f"<th>{html.escape(c)}</th>" for c in labels_jp)+"</tr>"
matrix_rows=""
for i,name in enumerate(labels_jp):
    cells="".join(
        f'<td class="{"diag" if i==j else ""}">{int(cm[i,j])}</td>'
        for j in order
    )
    matrix_rows+=f"<tr><th>{html.escape(name)}</th>{cells}</tr>"
matrix=(
    '<h3>混同行列（画像の枚数）</h3><div style="overflow-x:auto">'
    '<table class="matrix"><thead>'+matrix_header+'</thead><tbody>'
    +matrix_rows+'</tbody></table></div>'
    '<p class="note">灰色のマスが正しく分類できた枚数です。'
    'それ以外のマスを見ると、どの病気同士を間違えたか分かります。</p>'
)
display(HTML(style+'<div class="leaf-report">'
   +f'<h3>testの評価結果（{len(y_true):,}枚）</h3>'+summary+table+matrix+'</div>'))


### 結果を見て考えてみよう

1. **Recallが最も低い病気はどれですか？** その病気の画像を、AIはどの病気と間違えることが多いでしょうか。混同行列も使って確かめましょう。

2. **ACCが高いものがいいモデルでしょうか？** 病気の見逃しを減らしたいとき、ACCだけでAIを評価してよいでしょうか。

3. **PrecisionとRecallのどちらを重視しますか？** 病気を見逃すことと、健康な葉を病気と誤判定することを比べて理由を説明しましょう。

4. このtest画像はデータセットに含まれる画像です。**別の条件で撮影した画像**でも、同じ精度が得られると思いますか。理由を考えましょう。


## 7. 学習したAIで葉の画像を判別してみよう

[Library Hiroba](https://pypi.org/project/library-hiroba/)のフォームで、**「test画像からランダムに選ぶ」**か**「自分の画像をアップロードする」**を選んで判別します。

- **test画像**：AIの予測と、画像の正解ラベルを見比べられます。
- **アップロード画像**：手元のJPEG/PNGを使えます。先に［画像を選択］でファイルを選び、モードを「アップロード画像」にして［判別する］を押してください。

※画像のアップロード操作のみColab向けの部品を使用します。フォーム・判別結果・メッセージの表示にはLibrary Hirobaを使用します。


In [ ]:
from library_hiroba import ui
from IPython.display import display
import html as htmlmod

def predict_image(pil):
    rgb=pil.convert("RGB").resize((SIZE,SIZE),Image.Resampling.BILINEAR)
    arr=np.asarray(rgb,dtype=np.uint8).copy()
    x=torch.from_numpy(arr).permute(2,0,1).float().div_(255).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.inference_mode():
        with torch.autocast(device_type="cuda",dtype=torch.float16,
                            enabled=(DEVICE.type=="cuda")):
            scores=torch.softmax(model(x),dim=1)[0].float().cpu().numpy()
    return int(scores.argmax()),scores

def result_html(pil,true_label=None,source="画像"):
    pred,prob=predict_image(pil)
    preview=pil.copy().convert("RGB")
    preview.thumbnail((420,350))
    buf=io.BytesIO()
    preview.save(buf,format="PNG")
    src=base64.b64encode(buf.getvalue()).decode("ascii")
    name=htmlmod.escape(JP.get(classes[pred],classes[pred]))
    status=""
    if true_label is not None:
        actual=htmlmod.escape(JP.get(true_label,true_label))
        mark="正解" if classes[pred]==true_label else "不正解"
        status=f"<p>正解ラベル：<b>{actual}</b>（{mark}）</p>"
    rows="".join(
        f'<tr><td style="padding:6px;border-bottom:1px solid #ddd">{htmlmod.escape(JP.get(c,c))}</td>'
        f'<td style="padding:6px;text-align:right;border-bottom:1px solid #ddd">{float(prob[i])*100:.1f}%</td></tr>'
        for i,c in enumerate(classes))
    return (
        '<div style="max-width:520px;font-family:system-ui,sans-serif;line-height:1.6;color:#222">'
        f'<h3 style="margin:0 0 10px">{htmlmod.escape(source)}</h3>'
        f'<img src="data:image/png;base64,{src}" alt="葉の画像" '
        'style="width:auto;max-width:100%;max-height:340px;object-fit:contain;display:block;margin:8px 0">'
        f'{status}<p>AIの予測：<b>{name}</b></p>'
        '<p>各クラスの予測スコア</p>'
        f'<table style="border-collapse:collapse;width:100%">{rows}</table>'
        '<p style="font-size:12px;color:#555">※予測スコアは病気の確定診断を表すものではありません。</p>'
        '</div>'
    )


In [ ]:
# Library Hirobaのフォームを使用（校則チャットボットの新UI版と同じ方式）
# Colabでのファイル選択にはipywidgets.FileUploadだけを併用します。
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

MODE_TEST="test画像からランダムに選ぶ"
MODE_UPLOAD="アップロード画像を判別する"

file_upload=widgets.FileUpload(
    accept=".png,.jpg,.jpeg",multiple=False,
    description="画像を選択（JPEG/PNG）")
file_upload.layout.width="270px"
result_view=widgets.HTML()

def update_result(component):
    # 参考Notebookと同様、Colabでは表示エリアのHTMLを更新する。
    # ui.formの戻り値は空にして結果を重複表示させない。
    result_view.value=component._repr_html_()

def run_classifier(mode):
    try:
        if mode==MODE_TEST:
            row=random.choice(test_rows)
            with Image.open(root/row["image"]) as im:
                html=result_html(im.copy(),true_label=row["label"],source="testデータ")
        elif mode==MODE_UPLOAD:
            data=file_upload.value
            if not data:
                update_result(ui.alert("先に［画像を選択］でJPEG/PNG画像を選んでください。",kind="warning"))
                return ui.html("")
            item=(next(iter(data.values())) if isinstance(data,dict) else data[0])
            raw=(item["content"] if isinstance(item,dict) else item.content)
            if len(raw)>15*1024*1024:
                raise ValueError("15MB以下の画像を選択してください")
            with Image.open(io.BytesIO(bytes(raw))) as im:
                html=result_html(im.copy(),source="アップロード画像")
        else:
            return ui.alert("判別方法を選んでください。",kind="warning")
        update_result(ui.html(html))
    except Exception as error:
        update_result(ui.alert(f"画像を判別できませんでした：{error}",kind="danger"))
    return ui.html("")

display(ui.card("トマトの葉を判別", "判別方法を選び、［判別する］を押してください。"))
display(ui.alert("アップロード画像を使う場合は、最初に下のボタンで画像を選択します。"))
display(file_upload)
display(ui.form(
    run_classifier,
    ui.field("mode",label="判別方法",kind="choice",
             choices=[MODE_TEST,MODE_UPLOAD]),
    title="画像の判別",
    submit_label="判別する",
    pending=None
))
display(result_view)


## 8. もっと性能を上げるには？（発展）

今回のCNNは、**短時間でAIの学習・評価を体験するための簡単なモデル**です。性能を高める方法はいくつもあります。

**① CNNの構造を改良する**

畳み込み層を増やす、フィルタ数を変える、あるいは画像分類で実績のある **ResNet** などを試す方法があります。高性能になる可能性はありますが、データ量や計算量によって最適なモデル構造も変わります。

**② データを改良する**

撮影場所、背景、葉の向き、明るさが異なる画像を集めたり、学習データだけに画像の反転・明るさの変化などを加えたりすることで、多様な画像に対応したAIにできます。画像の枚数だけでなく、データの質も大切です。

**③ ハイパーパラメータ探索を広げる**

今回はOptunaを少数試行にしました。試行数・交差検証の分割数・最大エポック数を増やし、学習率やDropoutなどを丁寧に比較できます。ただし、学習時間は長くなります。


---

### 教材・データの注意
元のデータでは完全一致画像の重複を除去していますが、同じ葉の別撮影画像まで分離できているとは限りません。また、実際の畑・家庭菜園の画像で同じ精度が出るとは限りません。

**データ出典**：Pandian & Gopal (2019), Mendeley Data, DOI: 10.17632/tywbtsjrjv.1
